In [1]:
import pandas as pd
import readability
import syntok.segmenter as segmenter
from evaluate import load
import numpy as np
bertscore = load("bertscore")

/Users/carlosivan/miniconda3/envs/bertscore/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
version_modelo = "Qwen3-0.6B-01"

In [3]:
REFERENCE_COL = 'pls' 
prediction_col = f'pls_{version_modelo}'

READABILITY_METRICS = [
    'Coleman-Liau', 
    'FleschReadingEase', 
    'GunningFogIndex', 
    'SMOGIndex', 
    'Kincaid', 
    'DaleChallIndex'
]

In [ ]:
data = pd.read_csv(f'../datos/resumenes_generados/resultados_{version_modelo}.csv', sep=';')

In [ ]:
def calculate_readability_metrics(text):
    results = {metric: np.nan for metric in READABILITY_METRICS}
    
    if not isinstance(text, str) or not text.strip():
        return results

    try:
        tokenized = '\n\n'.join(
            '\n'.join(' '.join(token.value for token in sentence) 
                      for sentence in paragraph) 
            for paragraph in segmenter.analyze(text)
        )
        measures = readability.getmeasures(tokenized, lang='en')
        
        grades = measures.get('readability grades')
        if grades:
            for metric in READABILITY_METRICS:
                if metric in grades:
                    results[metric] = round(grades[metric],3)
        return results
    except Exception:
        return results

In [7]:
readability_results = data[prediction_col].apply(calculate_readability_metrics)

In [8]:
readability_df = pd.json_normalize(readability_results)
data = data.join(readability_df)

In [9]:
predictions = data[prediction_col].fillna("").tolist()
references = data[REFERENCE_COL].fillna("").tolist()

In [10]:
bert_results = bertscore.compute(
        predictions=predictions,
        references=references,
        model_type="allenai/longformer-large-4096-finetuned-triviaqa",
        verbose=True
    )
    
data['bertscore_f1'] = bert_results['f1']

calculating scores...
computing bert embedding.


100%|██████████| 7/7 [10:17<00:00, 88.23s/it] 


computing greedy matching.


100%|██████████| 4/4 [00:01<00:00,  2.44it/s]


done in 139513.42 seconds, 0.00 sentences/sec


In [ ]:
output_filename = f'../datos/resultados_metricas/readability_bertscore_{version_modelo}.csv'

In [12]:
data.to_csv(output_filename, sep=';', index=False)